In [ ]:
# ============================================================
# Part C: forward-backward experiment code
# ============================================================

# Assumes Part A and Part B have already been run so the following exist:
#   - apply_precond, undo_precond, q_sample, p_sample_step
#   - classify_ddpm_raw
#   - ddpm_artifacts
#   - clf_artifacts


# ------------------------------------------------------------
# Data helpers for experiment batches
# ------------------------------------------------------------

def load_raw_mnist_tensors(root: str = "./data", train: bool = False):
    """
    Returns:
      x01: [N,1,28,28] in [0,1]
      y:   [N]
    """
    ds = datasets.MNIST(root=root, train=train, download=True)
    x01 = ds.data.float().unsqueeze(1) / 255.0
    y = ds.targets.long()
    return x01, y


def get_class_subset_pool(class_subset, root: str = "./data", split: str = "test"):
    train = (split == "train")
    x01, y = load_raw_mnist_tensors(root=root, train=train)

    class_subset = tuple(sorted(int(c) for c in class_subset))
    mask = torch.zeros_like(y, dtype=torch.bool)
    for c in class_subset:
        mask |= (y == c)

    return x01[mask], y[mask]


def prepare_experiment_batch(
    class_subset,
    n_samples: int,
    root: str = "./data",
    split: str = "test",
    device: torch.device = device,
):
    """
    Sample once from the restricted dataset.
    Reusing this exact batch across all timesteps makes the curves easier to compare.
    """
    x_pool01, y_pool = get_class_subset_pool(class_subset=class_subset, root=root, split=split)
    idx = torch.randperm(x_pool01.size(0))[:n_samples]

    batch = {
        "x01": x_pool01[idx].to(device),
        "y": y_pool[idx].to(device),
        "class_subset": tuple(sorted(int(c) for c in class_subset)),
        "split": split,
    }
    return batch


def make_noise_bank(timesteps, batch_shape, device: torch.device = device):
    """
    Optional helper: use the same noise for multiple DDPM variants so
    identity-vs-M comparisons are cleaner.
    """
    return {int(t): torch.randn(batch_shape, device=device) for t in timesteps}


# ------------------------------------------------------------
# Forward-backward DDPM
# ------------------------------------------------------------

@torch.no_grad()
def reverse_from_xt(ddpm_artifacts: dict, x_t: torch.Tensor, t_start: int) -> torch.Tensor:
    """
    Start from x_t and run the reverse chain back to x_0 in DDPM training space.
    """
    model = ddpm_artifacts["model"]
    diffusion = ddpm_artifacts["diffusion"]

    model.eval()
    x = x_t.clone()
    for i in range(int(t_start), -1, -1):
        x = p_sample_step(model, x, i, diffusion)
    return x


def subset_membership_mask(preds: torch.Tensor, class_subset) -> torch.Tensor:
    mask = torch.zeros_like(preds, dtype=torch.bool)
    for c in class_subset:
        mask |= (preds == int(c))
    return mask


@torch.no_grad()
def run_forward_backward_at_t(
    ddpm_artifacts: dict,
    clf_artifacts: dict,
    experiment_batch: dict,
    t: int,
    noise: torch.Tensor | None = None,
    store_images: bool = False,
):
    """
    One timestep evaluation:
      x_raw in [0,1] -> [-1,1]
      -> DDPM forward to time t
      -> DDPM reverse back to x0_hat
      -> back to raw [-1,1]
      -> classifier evaluation
    """
    x01 = experiment_batch["x01"]
    y = experiment_batch["y"]
    class_subset = experiment_batch["class_subset"]

    x_raw = x01 * 2.0 - 1.0
    x0_ddpm = apply_precond(x_raw, ddpm_artifacts["precond"])

    if noise is None:
        noise = torch.randn_like(x0_ddpm)

    t_batch = torch.full((x0_ddpm.size(0),), int(t), device=x0_ddpm.device, dtype=torch.long)
    x_t = q_sample(x0_ddpm, t_batch, noise, ddpm_artifacts["diffusion"])
    x0_hat_ddpm = reverse_from_xt(ddpm_artifacts, x_t, int(t))
    x0_hat_raw = undo_precond(x0_hat_ddpm, ddpm_artifacts["precond"]).clamp(-1, 1)

    preds, conf, probs = classify_ddpm_raw(clf_artifacts, x0_hat_raw)
    within_subset = subset_membership_mask(preds, class_subset)

    out = {
        "t": int(t),
        "within_subset_prop": within_subset.float().mean().item(),
        "preds": preds.detach().cpu(),
        "confidence": conf.detach().cpu(),
        "prediction_hist": torch.bincount(preds.detach().cpu(), minlength=10).tolist(),
        "source_labels": y.detach().cpu(),
    }

    if store_images:
        out["source_raw"] = x_raw.detach().cpu()
        out["x_t_ddpm"] = x_t.detach().cpu()
        out["recon_raw"] = x0_hat_raw.detach().cpu()

    return out


@torch.no_grad()
def run_forward_backward_experiment(
    ddpm_artifacts: dict,
    clf_artifacts: dict,
    class_subset,
    timesteps,
    n_samples: int = 512,
    root: str = "./data",
    split: str = "test",
    experiment_batch: dict | None = None,
    noise_bank: dict | None = None,
    store_details: bool = False,
):
    """
    Main experiment:
      - choose class subset S
      - sample uniformly from MNIST restricted to S
      - for each timestep t:
          forward-noise to t
          reverse-denoise to x0_hat
          classify x0_hat
          record proportion whose predicted label lies in S
    """
    if experiment_batch is None:
        experiment_batch = prepare_experiment_batch(
            class_subset=class_subset,
            n_samples=n_samples,
            root=root,
            split=split,
            device=device,
        )

    summary = []
    details = {}

    for t in timesteps:
        t = int(t)
        noise_t = None if noise_bank is None else noise_bank[t]

        out = run_forward_backward_at_t(
            ddpm_artifacts=ddpm_artifacts,
            clf_artifacts=clf_artifacts,
            experiment_batch=experiment_batch,
            t=t,
            noise=noise_t,
            store_images=store_details,
        )

        summary.append({
            "t": out["t"],
            "within_subset_prop": out["within_subset_prop"],
            "mean_confidence": float(out["confidence"].float().mean().item()),
            "prediction_hist": out["prediction_hist"],
        })

        if store_details:
            details[t] = out

    result = {
        "summary": summary,
        "details": details,
        "class_subset": experiment_batch["class_subset"],
        "n_samples": experiment_batch["x01"].size(0),
        "split": experiment_batch["split"],
        "precond_kind": ddpm_artifacts["precond"]["kind"],
        "timesteps": [int(t) for t in timesteps],
    }
    return result


# ------------------------------------------------------------
# Plotting / visualization
# ------------------------------------------------------------

def plot_within_subset_curve(experiment_result: dict, kind: str = "line", label: str | None = None):
    ts = [row["t"] for row in experiment_result["summary"]]
    ys = [row["within_subset_prop"] for row in experiment_result["summary"]]

    plt.figure(figsize=(7, 4))
    if kind == "bar":
        plt.bar(ts, ys)
    else:
        plt.plot(ts, ys, marker="o", label=label if label is not None else experiment_result["precond_kind"])
        if label is not None:
            plt.legend()

    plt.xlabel("diffusion time step")
    plt.ylabel("proportion predicted in subset")
    plt.ylim(0.0, 1.0)
    plt.title(
        f"Subset {list(experiment_result['class_subset'])} | "
        f"preconditioner = {experiment_result['precond_kind']}"
    )
    plt.grid(alpha=0.3)
    plt.show()


def plot_experiment_comparison(results_by_name: dict):
    plt.figure(figsize=(7, 4))
    for name, experiment_result in results_by_name.items():
        ts = [row["t"] for row in experiment_result["summary"]]
        ys = [row["within_subset_prop"] for row in experiment_result["summary"]]
        plt.plot(ts, ys, marker="o", label=name)

    any_result = next(iter(results_by_name.values()))
    plt.xlabel("diffusion time step")
    plt.ylabel("proportion predicted in subset")
    plt.ylim(0.0, 1.0)
    plt.title(f"Subset {list(any_result['class_subset'])}")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


@torch.no_grad()
def show_forward_backward_examples(experiment_result: dict, t: int, n_show: int = 8):
    """
    Requires store_details=True when running the experiment.
    """
    t = int(t)
    d = experiment_result["details"][t]

    source = (d["source_raw"][:n_show].clamp(-1, 1) + 1.0) / 2.0
    recon = (d["recon_raw"][:n_show].clamp(-1, 1) + 1.0) / 2.0

    grid_source = make_grid(source, nrow=n_show, padding=4)
    grid_recon = make_grid(recon, nrow=n_show, padding=4)

    plt.figure(figsize=(14, 3))
    plt.subplot(1, 2, 1)
    plt.axis("off")
    plt.title("source images")
    plt.imshow(grid_source.permute(1, 2, 0).squeeze(), cmap="gray")

    plt.subplot(1, 2, 2)
    plt.axis("off")
    plt.title(f"reconstructions at t={t}")
    plt.imshow(grid_recon.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.show()

    print("source labels :", d["source_labels"][:n_show].tolist())
    print("pred labels   :", d["preds"][:n_show].tolist())
    print("confidences   :", [round(x, 3) for x in d["confidence"][:n_show].tolist()])


# ------------------------------------------------------------
# Example usage
# ------------------------------------------------------------
# 1) Train or load both DDPM variants
# ddpm_identity = load_ddpm_checkpoint("ddpm_identity.pt")
# ddpm_special  = load_ddpm_checkpoint("ddpm_special.pt")
#
# 2) Train or load classifier
# clf_artifacts = load_classifier_checkpoint("mnist_classifier.pt")
#
# 3) Prepare one shared source batch and one shared noise bank
# class_subset = {3, 8}
# timesteps = [0, 25, 50, 100, 150, 200, 300, 399]
#
# batch = prepare_experiment_batch(
#     class_subset=class_subset,
#     n_samples=512,
#     split="test",
# )
# noise_bank = make_noise_bank(timesteps=timesteps, batch_shape=batch["x01"].shape, device=device)
#
# 4) Run the experiment for each DDPM variant
# result_identity = run_forward_backward_experiment(
#     ddpm_artifacts=ddpm_identity,
#     clf_artifacts=clf_artifacts,
#     class_subset=class_subset,
#     timesteps=timesteps,
#     experiment_batch=batch,
#     noise_bank=noise_bank,
#     store_details=True,
# )
#
# result_special = run_forward_backward_experiment(
#     ddpm_artifacts=ddpm_special,
#     clf_artifacts=clf_artifacts,
#     class_subset=class_subset,
#     timesteps=timesteps,
#     experiment_batch=batch,
#     noise_bank=noise_bank,
#     store_details=True,
# )
#
# 5) Plot
# plot_within_subset_curve(result_identity)
# plot_within_subset_curve(result_special)
# plot_experiment_comparison({
#     "identity": result_identity,
#     "special_M": result_special,
# })
#
# 6) Inspect reconstructions at one t
# show_forward_backward_examples(result_special, t=150, n_show=8)